# Indian E-commerce Returns & Fulfilment Intelligence
## Exploratory Data Analysis (EDA)

**Objective:** Analyse item sales, product returns, refunds, and delivery performance to identify useful business insights.

**Input:** `cleaned_ecommerce_order_items.csv` exported by `Data_Cleaning_and_Preparation.ipynb`.

**Metric notes:** Orders are counted using distinct `order_id`; item-level metrics use order-item rows. Missing delivery or return flags are excluded from the relevant rate denominator. Recorded refund amounts are not proof that a refund was successfully settled.

### 1. Project Overview

This notebook explores the cleaned dataset to answer:
- Which product categories generate the most recorded item sales?
- Which categories and reasons are associated with returns?
- How well are delivery promises being met?
- How do sales, returns, and late deliveries change over time?

### 2. Load and Understand Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_PATH = Path("cleaned_ecommerce_order_items.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Cleaned CSV not found. Run Data_Cleaning_and_Preparation.ipynb first "
        "and keep the exported CSV beside this notebook."
    )

df = pd.read_csv(DATA_PATH)
df.columns = (
    df.columns.str.strip().str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

date_cols = ["order_datetime", "shipped_datetime", "delivered_datetime"]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

numeric_cols = [
    "seller_rating", "quantity", "mrp", "discount_pct", "selling_price",
    "item_total", "promised_delivery_days", "actual_delivery_days",
    "is_late_delivery", "is_returned", "refund_amount", "refund_days",
    "is_plus_member"
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Dataset shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("Data Type"))
display(df.describe(include="all").T)

### 3. Business KPIs

In [ ]:
total_orders = df["order_id"].nunique() if "order_id" in df.columns else 0
total_items = len(df)
recorded_item_sales = df["item_total"].sum(min_count=1) if "item_total" in df.columns else float("nan")

# Average recorded order value: sum item_total within each order, then average across orders.
if {"order_id", "item_total"}.issubset(df.columns):
    order_values = df.groupby("order_id", dropna=True)["item_total"].sum(min_count=1)
    average_order_value = order_values.mean()
else:
    average_order_value = float("nan")

if "is_returned" in df.columns:
    returned = df["is_returned"].eq(1)
    known_returns = df["is_returned"].isin([0, 1])
    returned_count = int(returned.sum())
    return_rate = (returned.sum() / known_returns.sum() * 100) if known_returns.sum() else float("nan")
else:
    returned_count, return_rate = 0, float("nan")

if "is_late_delivery" in df.columns:
    known_late = df["is_late_delivery"].isin([0, 1])
    late_count = int(df["is_late_delivery"].eq(1).sum())
    late_rate = (late_count / known_late.sum() * 100) if known_late.sum() else float("nan")
else:
    late_count, late_rate = 0, float("nan")

kpis = pd.DataFrame({
    "Metric": [
        "Total Orders", "Total Order Items", "Recorded Item Sales",
        "Average Recorded Order Value", "Returned Items", "Return Rate (%)",
        "Late Delivery Items with Known Flag", "Late Delivery Rate (%)"
    ],
    "Value": [
        total_orders, total_items, recorded_item_sales, average_order_value,
        returned_count, return_rate, late_count, late_rate
    ]
})
display(kpis)

### 4. Univariate Analysis

In [ ]:
# Categorical distributions
for col in ["order_status", "item_status"]:
    if col in df.columns and df[col].notna().any():
        counts = df[col].fillna("Missing").value_counts()
        ax = counts.plot(kind="bar", figsize=(9, 4), title=f"{col.replace('_', ' ').title()} Distribution")
        ax.set_xlabel(col.replace("_", " ").title())
        ax.set_ylabel("Order-item rows")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

# Numeric distributions
for col in ["item_total", "discount_pct", "actual_delivery_days"]:
    if col in df.columns and df[col].notna().any():
        plt.figure(figsize=(8, 4))
        sns.histplot(df[col].dropna(), bins=30, kde=False)
        plt.title(f"Distribution of {col.replace('_', ' ').title()}")
        plt.xlabel(col.replace("_", " ").title())
        plt.ylabel("Frequency")
        plt.tight_layout()
        plt.show()

### 5. Bivariate Analysis

In [ ]:
# Recorded item sales by product category
if {"category", "item_total"}.issubset(df.columns):
    category_sales = df.groupby("category", dropna=False)["item_total"].sum().sort_values(ascending=False)
    display(category_sales.to_frame("Recorded Item Sales"))
    category_sales.plot(kind="bar", figsize=(10, 4), title="Recorded Item Sales by Category")
    plt.xlabel("Category")
    plt.ylabel("Sum of item_total")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

# Return rate by category; exclude unknown return flags from the denominator
if {"category", "is_returned"}.issubset(df.columns):
    known = df[df["is_returned"].isin([0, 1])]
    if not known.empty:
        category_return = known.groupby("category", dropna=False)["is_returned"].mean().mul(100).sort_values(ascending=False)
        display(category_return.to_frame("Return Rate (%)"))
        category_return.plot(kind="bar", figsize=(10, 4), title="Return Rate by Category")
        plt.ylabel("Return Rate (%)")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

# Discount vs item sales
if {"discount_pct", "item_total"}.issubset(df.columns):
    plot_data = df[["discount_pct", "item_total"]].dropna()
    if not plot_data.empty:
        plt.figure(figsize=(7, 4))
        sns.scatterplot(data=plot_data, x="discount_pct", y="item_total", alpha=0.35)
        plt.title("Discount Percentage vs. Item Total")
        plt.tight_layout()
        plt.show()

### 6. Multivariate Analysis

In [ ]:
# Return rate across category and region
if {"category", "region", "is_returned"}.issubset(df.columns):
    known = df[df["is_returned"].isin([0, 1])]
    if not known.empty:
        return_pivot = known.pivot_table(
            index="category", columns="region", values="is_returned",
            aggfunc="mean"
        ) * 100
        display(return_pivot.round(2))
        plt.figure(figsize=(12, 6))
        sns.heatmap(return_pivot, annot=False, cmap="YlOrRd")
        plt.title("Return Rate (%) by Category and Region")
        plt.tight_layout()
        plt.show()

# Late delivery rate by courier and region
if {"courier_partner", "region", "is_late_delivery"}.issubset(df.columns):
    known = df[df["is_late_delivery"].isin([0, 1])]
    if not known.empty:
        late_pivot = known.pivot_table(
            index="courier_partner", columns="region", values="is_late_delivery",
            aggfunc="mean"
        ) * 100
        display(late_pivot.round(2))
        plt.figure(figsize=(12, 5))
        sns.heatmap(late_pivot, annot=False, cmap="YlOrRd")
        plt.title("Late Delivery Rate (%) by Courier and Region")
        plt.tight_layout()
        plt.show()

### 7. Correlation Analysis

In [ ]:
corr_cols = [
    col for col in [
        "quantity", "mrp", "discount_pct", "selling_price", "item_total",
        "seller_rating", "promised_delivery_days", "actual_delivery_days",
        "refund_amount", "refund_days"
    ]
    if col in df.columns and pd.api.types.is_numeric_dtype(df[col])
]

corr_data = df[corr_cols].dropna(how="all") if corr_cols else pd.DataFrame()
if len(corr_cols) >= 2 and not corr_data.empty:
    correlation = corr_data.corr(numeric_only=True)
    display(correlation.round(2))
    plt.figure(figsize=(10, 7))
    sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric columns with data for correlation analysis.")

# Correlation shows association, not causation.

### 8. Outlier Analysis

In [ ]:
outlier_cols = ["item_total", "selling_price", "actual_delivery_days"]

for col in outlier_cols:
    if col not in df.columns:
        continue
    values = df[col].dropna()
    if values.empty:
        continue

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_count = int(((values < lower) | (values > upper)).sum())

    print(f"{col}: {outlier_count:,} potential outliers (IQR method)")
    plt.figure(figsize=(8, 3))
    sns.boxplot(x=values)
    plt.title(f"Potential Outliers in {col.replace('_', ' ').title()}")
    plt.tight_layout()
    plt.show()

# Outliers are flagged for review, not automatically removed.

### 9. Product and Return Analysis

In [ ]:
# Sales by sub-category
if {"sub_category", "item_total"}.issubset(df.columns):
    subcategory_sales = df.groupby("sub_category", dropna=False)["item_total"].sum().sort_values(ascending=False).head(15)
    display(subcategory_sales.to_frame("Recorded Item Sales"))
    subcategory_sales.sort_values().plot(kind="barh", figsize=(9, 5), title="Top 15 Sub-categories by Recorded Item Sales")
    plt.xlabel("Sum of item_total")
    plt.tight_layout()
    plt.show()

# Return reasons and statuses
for col in ["return_reason", "return_status"]:
    if col in df.columns:
        counts = df[col].dropna().value_counts()
        if not counts.empty:
            display(counts.to_frame("Order-item Rows"))
            counts.head(10).plot(kind="bar", figsize=(8, 4), title=f"{col.replace('_', ' ').title()} Distribution")
            plt.ylabel("Order-item rows")
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            plt.show()

# Recorded refunds: values are not confirmation of refund settlement
if "refund_amount" in df.columns:
    refund_rows = df[df["refund_amount"].notna()]
    print("Rows with a recorded refund amount:", len(refund_rows))
    print("Total recorded refund amount:", round(refund_rows["refund_amount"].sum(), 2))
    if "refund_mode" in refund_rows.columns:
        display(refund_rows["refund_mode"].value_counts(dropna=False).to_frame("Rows"))
    if "refund_days" in refund_rows.columns:
        print("Average recorded refund days:", round(refund_rows["refund_days"].mean(), 2))

### 10. Delivery and Fulfilment Analysis

In [ ]:
# Compare promised vs actual delivery days
if {"promised_delivery_days", "actual_delivery_days"}.issubset(df.columns):
    delivery = df[["promised_delivery_days", "actual_delivery_days"]].dropna()
    if not delivery.empty:
        display(delivery.describe().round(2))
        plt.figure(figsize=(7, 5))
        sns.scatterplot(data=delivery, x="promised_delivery_days", y="actual_delivery_days", alpha=0.35)
        max_day = max(delivery.max().max(), 1)
        plt.plot([0, max_day], [0, max_day], linestyle="--")
        plt.title("Promised vs. Actual Delivery Days")
        plt.tight_layout()
        plt.show()

# Late delivery rate by courier; only known flags in denominator
if {"courier_partner", "is_late_delivery"}.issubset(df.columns):
    known = df[df["is_late_delivery"].isin([0, 1])]
    if not known.empty:
        courier_late = (known.groupby("courier_partner", dropna=False)["is_late_delivery"].mean() * 100).sort_values(ascending=False)
        display(courier_late.to_frame("Late Delivery Rate (%)"))
        courier_late.plot(kind="bar", figsize=(8, 4), title="Late Delivery Rate by Courier")
        plt.ylabel("Late Delivery Rate (%)")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

### 11. Monthly Trend Analysis

In [ ]:
if "order_datetime" in df.columns:
    monthly_df = df.dropna(subset=["order_datetime"]).copy()
    monthly_df["order_month"] = monthly_df["order_datetime"].dt.to_period("M").dt.to_timestamp()

    monthly_sales = monthly_df.groupby("order_month")["item_total"].sum(min_count=1) if "item_total" in monthly_df.columns else pd.Series(dtype=float)
    monthly_orders = monthly_df.groupby("order_month")["order_id"].nunique() if "order_id" in monthly_df.columns else pd.Series(dtype=float)

    if not monthly_sales.empty:
        monthly_sales.plot(figsize=(10, 4), marker="o", title="Monthly Recorded Item Sales")
        plt.ylabel("Sum of item_total")
        plt.tight_layout()
        plt.show()

    if not monthly_orders.empty:
        monthly_orders.plot(figsize=(10, 4), marker="o", title="Monthly Order Volume")
        plt.ylabel("Distinct orders")
        plt.tight_layout()
        plt.show()

    if "is_returned" in monthly_df.columns:
        known_returns = monthly_df[monthly_df["is_returned"].isin([0, 1])]
        if not known_returns.empty:
            monthly_return_rate = known_returns.groupby("order_month")["is_returned"].mean() * 100
            monthly_return_rate.plot(figsize=(10, 4), marker="o", title="Monthly Return Rate")
            plt.ylabel("Return Rate (%)")
            plt.tight_layout()
            plt.show()

    if "is_late_delivery" in monthly_df.columns:
        known_late = monthly_df[monthly_df["is_late_delivery"].isin([0, 1])]
        if not known_late.empty:
            monthly_late_rate = known_late.groupby("order_month")["is_late_delivery"].mean() * 100
            monthly_late_rate.plot(figsize=(10, 4), marker="o", title="Monthly Late Delivery Rate")
            plt.ylabel("Late Delivery Rate (%)")
            plt.tight_layout()
            plt.show()
else:
    print("order_datetime is unavailable; monthly trends cannot be calculated.")

### 12. Key Business Insights and Recommendations

In [ ]:
findings = []

if {"category", "item_total"}.issubset(df.columns):
    sales_by_category = df.groupby("category")["item_total"].sum(min_count=1).dropna()
    if not sales_by_category.empty:
        top_category = sales_by_category.idxmax()
        findings.append(
            f"Highest recorded item sales category: {top_category} "
            f"({sales_by_category.max():,.2f})."
        )

if {"category", "is_returned"}.issubset(df.columns):
    known_returns = df[df["is_returned"].isin([0, 1])]
    if not known_returns.empty:
        rates = known_returns.groupby("category")["is_returned"].mean().dropna() * 100
        if not rates.empty:
            highest_return_category = rates.idxmax()
            findings.append(
                f"Highest observed return-rate category: {highest_return_category} "
                f"({rates.max():.1f}%). Check category sample sizes before acting."
            )

if {"return_reason", "is_returned"}.issubset(df.columns):
    reasons = df.loc[df["is_returned"].eq(1), "return_reason"].dropna().value_counts()
    if not reasons.empty:
        findings.append(
            f"Most common recorded return reason: {reasons.index[0]} "
            f"({int(reasons.iloc[0]):,} item rows)."
        )

if {"courier_partner", "is_late_delivery"}.issubset(df.columns):
    known_late = df[df["is_late_delivery"].isin([0, 1])]
    if not known_late.empty:
        rates = known_late.groupby("courier_partner")["is_late_delivery"].mean().dropna() * 100
        if not rates.empty:
            findings.append(
                f"Highest observed late-delivery rate among couriers: {rates.idxmax()} "
                f"({rates.max():.1f}%). Review valid-record counts before comparison."
            )

if findings:
    for number, finding in enumerate(findings, start=1):
        print(f"{number}. {finding}")
else:
    print("No findings could be generated from the available columns and valid records.")

print("\nRecommendations should be based on these results, record counts, and business context. "
      "Associations do not by themselves prove causes.")